# BloomML — Kaggle T4 × 2 (research only)

This notebook couples Kaggle to **MichalMatu/bloomml** by cloning the chosen research branch, recording the *exact* code SHA and saving an artifact manifest. **The simulator currently computes on CPU**, while both T4 cards are verified by a real CUDA smoke test. Do not report GPU physics until a batched solver with CPU parity has been implemented.

Requirements: Kaggle private notebook; Settings → Accelerator → GPU T4 x2; Internet ON for the initial clone; no tokens/API keys; outputs stay private. Stop an existing Ollama notebook if it occupies the GPUs.

In [ ]:
import os, subprocess
from pathlib import Path

REPOSITORY = "https://github.com/MichalMatu/bloomml.git"
BRANCH = "research/simulator-fidelity-kaggle-20261103"
# Optional exact commit pin from a verified checkpoint; leave empty to resolve
# and record the latest BRANCH HEAD for this run.
PINNED_SHA = ""
SOURCE = Path("/kaggle/working/bloomml-source")
if SOURCE.exists():
    raise RuntimeError("Source folder already exists. Restart notebook to avoid stale code.")
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY, str(SOURCE)], check=True)
resolved = subprocess.check_output(["git", "-C", str(SOURCE), "rev-parse", "HEAD"], text=True).strip()
if PINNED_SHA:
    subprocess.run(["git", "-C", str(SOURCE), "checkout", "--detach", PINNED_SHA], check=True)
    resolved = subprocess.check_output(["git", "-C", str(SOURCE), "rev-parse", "HEAD"], text=True).strip()
    assert resolved == PINNED_SHA, "Source does not match requested pin"
assert len(resolved) == 40 and all(c in "0123456789abcdef" for c in resolved)
assert subprocess.check_output(["git", "-C", str(SOURCE), "status", "--porcelain"], text=True).strip() == ""
print("BLOOMML_SOURCE_SHA =", resolved)
print("BLOOMML_SOURCE_BRANCH =", BRANCH)

In [ ]:
import sys
sys.path.insert(0, str(SOURCE))
from tools.ml.kaggle_physics_runner import execute

# Both T4 are required. GPU math smoke runs on each device.
# The v2 synthetic thermodynamic solver still computes on CPU.
output_dir = Path("/kaggle/working/bloomml-run")
manifest = execute(
    output_dir=output_dir,
    source_dir=SOURCE,
    scenarios=4,
    steps=120,
    seed=1729,
    required_gpus=2,
)
print("T4 GPUs:", [d["name"] for d in manifest["accelerator"]["used_devices"]])
print("SOURCE SHA:", manifest["source_commit"])
print("CPU golden steps:", manifest["physics_summary"]["total_steps"])
print("Water residual kg:", manifest["physics_summary"]["max_water_residual_kg"])
print("GPU physics ready:", manifest["gpu_physics_ready"])
print("Manifest:", output_dir / "kaggle_manifest.json")

In [ ]:
import json
print(json.dumps({
    "source_commit": manifest["source_commit"],
    "physics_stage": manifest["stage"],
    "gpu_physics_ready": manifest["gpu_physics_ready"],
    "golden_digest": manifest["physics_summary"]["sha256"],
    "artifacts": list(manifest["artifacts"].keys()) + ["kaggle_manifest.json"],
}, indent=2))
print("Download /kaggle/working/bloomml-run/* as Kaggle notebook output, then keep results off public Git.")